# 🤖 Classification de CVs — Prédiction de `passed_next_stage`

Ce notebook entraîne plusieurs modèles de classification pour prédire si un CV passe à l'étape suivante du recrutement.

**Pipeline :**
1. Chargement & aperçu des données
2. Définition des features (numériques, catégorielles, texte)
3. Split train / test stratifié
4. Pré-traitement (StandardScaler + OneHotEncoder + TF-IDF)
5. Entraînement de 4 modèles avec SMOTE (gestion du déséquilibre)
6. Évaluation (ROC-AUC, F1, matrices de confusion)
7. Importance des features (meilleur modèle)

## 1. Imports

In [ ]:
import warnings

from matplotlib.pyplot import grid

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Versions
import sklearn, imblearn
print(f'pandas     {pd.__version__}')
print(f'numpy      {np.__version__}')
print(f'sklearn    {sklearn.__version__}')
print(f'imblearn   {imblearn.__version__}')
print(f'seaborn    {sns.__version__}')

## 2. Chargement & aperçu des données

In [ ]:
df = pd.read_csv('../data/cv_dataset.csv')   # adapter le chemin si nécessaire

print(f'Shape : {df.shape}')
df.head()

In [ ]:
print('Types de colonnes :')
print(df.dtypes)
print()
print('Valeurs manquantes :')
print(df.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution de la cible
counts = df['passed_next_stage'].value_counts()
axes[0].bar(['Refusé (0)', 'Sélectionné (1)'], counts.values,
            color=['#e74c3c', '#2ecc71'], alpha=0.85, edgecolor='white')
axes[0].set_title('Distribution de la variable cible')
axes[0].set_ylabel('Nombre de CVs')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Taux de sélection par rôle
role_rate = df.groupby('target_role')['passed_next_stage'].mean().sort_values()
role_rate.plot(kind='barh', ax=axes[1], color='steelblue', alpha=0.8)
axes[1].set_title('Taux de sélection par rôle')
axes[1].set_xlabel('Taux de sélection')
axes[1].axvline(df['passed_next_stage'].mean(), color='red', linestyle='--',
                label=f"Moyenne globale ({df['passed_next_stage'].mean():.1%})")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"Taux de sélection global : {df['passed_next_stage'].mean():.1%}")

## 3. Définition des features

In [ ]:
TARGET = 'passed_next_stage'
DROP_COLS = ['cv_id']

NUMERIC_FEATURES = [
    'age',
    'distance_ville_haute_km',
    'total_experience_years',
    'total_gap_months',
    'nb_gaps',
    'education_score',
    'number_of_experiences',
    'lang_fr',
    'lang_en',
    'lang_de',
    'lang_es',
    'lang_it',
    'lang_other_score_sum',
]

CATEGORICAL_FEATURES = [
    'target_role',
    'education_degree',
    'education_field',
    'education_school',
]

TEXT_SKILLS         = 'skills'
TEXT_CERTIFICATIONS = 'certifications'

X = df.drop(columns=DROP_COLS + [TARGET])
y = df[TARGET]

print(f'Features numériques    : {len(NUMERIC_FEATURES)}')
print(f'Features catégorielles : {len(CATEGORICAL_FEATURES)}')
print(f'Colonnes texte         : 2 (skills + certifications)')
print(f'Shape de X             : {X.shape}')
print(f'Shape de y             : {y.shape}')